<a href="https://colab.research.google.com/github/fanat503/Induction-Heads-Tinystories/blob/main/induction_heads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
device = 'cuda' if torch.cuda_available else 'cpu'

def load_checkpoint(path, model, device):
    state = torch.load(path, map_location=device)
    model.load_state_dict(state['model_state_dict'])
    model.eval()
    return model


def make_repeated_sequence(tokenizer, length=50, device='cuda'):
    random_tokens = torch.randint(100, 5000, (length,))
    repeated = torch.cat([random_tokens, random_tokens])
    return repeated.unsqueeze(0).to(device)


def collect_attention_all_layers(model, input_ids):
    attention_maps = {}

    with torch.no_grad():
        x = model.transformer.wte(input_ids) + \
            model.transformer.wpe(
                torch.arange(input_ids.size(1), device=input_ids.device)
            )

        for layer_idx, block in enumerate(model.transformer.h):
            x_norm = block.ln_1(x)

            attn_out, attn_weights = block.attn.forward_with_attention(x_norm)

            attention_maps[layer_idx] = attn_weights.cpu()
            x = x + attn_out
            x = x + block.mlp(block.ln_2(x))

    return attention_maps

def compute_induction_score(attention_maps, seq_len_half):
    scores = {}
    for layer in attention_maps:
        for head in range(12):
            ind_pat = []
            for t in range(seq_len_half, seq_len_half * 2):
                source = t - seq_len_half + 1
                ind_pat.append(
                    attention_maps[layer][0, head, t, source]
                )
            ind_score = torch.tensor(ind_pat).mean()
            scores[(layer, head)] = ind_score
    return scores


def compute_previous_token_score(attention_maps):
    Score = {}
    for layer in attention_maps:
      for head in range(12):
        prev_tok_pat = []
        for t in range(1, 100):
            prev_tok_pat.append(attention_maps[layer][0, head, t, t-1])
        Score[(layer, head)] = torch.tensor(prev_tok_pat).mean()
    return Score

def analyze_checkpoint(path, seq_len_half, model, device):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    step = checkpoint['step']
    torch.manual_seed(42)

    random_tokens = torch.randint(100, 5000, (seq_len_half,))
    repeated = torch.cat([random_tokens, random_tokens])
    input_ids = repeated.unsqueeze(0).to(device)
    attention_maps = collect_attention_all_layers(model, input_ids)

    ih_scores = compute_induction_score(attention_maps, seq_len_half)
    pth_scores = compute_previous_token_score(attention_maps)
    def print_top_scores(scores, name, top_n=10):
      sorted_scores = sorted(
        scores.items(),
        key=lambda x: x[1].item(),
        reverse=True
        )
      for i, (key, score) in enumerate(sorted_scores[:top_n]):
        print(f"  Layer {key[0]:2d}, Head {key[1]:2d}: {score:.4f}")

    result = {
        'induction': ih_scores,
        'previous_token': pth_scores,
        'step': step
    }

    print(f"\n Step {step}")
    print_top_scores(ih_scores, "Induction Score")
    print_top_scores(pth_scores, "Previous Token Score")

    return result

import matplotlib.pyplot as plt
import numpy as np

def visualisation(scores, n_layer, n_head, title, step):
  grid = np.zeros((n_layer, n_layer))
  for layer in range(n_layer):
    for head in range(n_head):
      grid[layer, head] = scores[layer, head]
  plt.imshow(grid)
  plt.colorbar(cmap='hot')
  plt.xlabel("Head")
  plt.ylabel("Layer")
  plt.title(title)
  plt.xticks(range(n_head))
  plt.yticks(range(n_layer))
  plt.savefig(f'*{title}_{step}.png', dpi=300)
  plt.show()

@torch.no_grad()
def test_induction_score(model, seq_len_half=50):
    model.eval()

    pattern = list(range(1, seq_len_half + 1))
    full_seq = pattern + pattern
    x = torch.tensor([full_seq])

    attention_maps = {}

    def make_hook(layer_idx):
        def hook(module, input, output):
            pass
        return hook

    tok_emb = model.transformer.wte(x)
    pos = torch.arange(0, x.shape[1]).unsqueeze(0)
    pos_emb = model.transformer.wpe(pos)
    hidden = tok_emb + pos_emb

    for i, block in enumerate(model.transformer.h):
        attn_out, att = block.attn.forward_with_attention(block.ln_1(hidden))
        attention_maps[i] = att.detach().cpu()

        hidden = hidden + attn_out
        hidden = hidden + block.mlp(block.ln_2(hidden))

    scores = compute_induction_score(attention_maps, seq_len_half)

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    for (layer, head), score in sorted_scores[:10]:
        print(f"  Layer {layer:2d}, Head {head:2d}: {score:.4f}")

    l6h4 = scores.get((6, 4), 0)
    print(f"\nL6H4: {l6h4:.4f}")

    if l6h4 > 0.1:
        print("*")
    elif l6h4 > 0.03:
        print("*")
    else:
        print("*")

    return scores

checkpoint = torch.load(
    '*',
    map_location='cpu'
)
model = GPT(GPTConfig())
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()